#문장의 유사도 분석
:두 개의 문장이 비슷한 것인지 또는 관련이 있는 것인지 분석(5가지)
-레벤슈타인 거리
-N-gram 
-코사인 유사도*(transformer)
-유클리언 유사도
-맨하튼 유사도

----
#레벤슈타인 거리(Levenshutein distance)
-두개의 문자열일 어느정도 다른지를 나타내는것으로 편집거리(edit distance)라고 부른다.(알고리즘)
-의학분야에서는 dna배열의 유사성을 판단할때도 사용.(순서에 민감함)

In [73]:
#길이 가지고 판단함. 포문
from lvenshtein import Lvenshtein
lv=Lvenshtein()

In [74]:
print(lv.calc_distance("가나다라",'가마바라'))
#2종류가 다르다:거리가 2
print(lv.calc_distance("가나다라",'가나다라'))
#0:동일하다

2
0


In [75]:
#신촌역과 가장 근접한 순서로 정렬

samples=['신발','마곡역','신천역','신천군','신촌역']
base=samples[4]#기본값,기준

r=sorted(samples, key=lambda n:lv.calc_distance(base,n))#최소 편집 갯수?
r

for n in r:
    print(lv.calc_distance(base,n),n)#틀린 갯수 알려줌
    #순서 민감 dna 몇개 틀렸는지 확률값
    

0 신촌역
1 신천역
2 신발
2 마곡역
2 신천군


---
#N-gram 유사도
-이웃한 N개의 문자
-서로 다른 2개의 문장을 N-gram으로 비교해보면 출현하는 단어의 종류와 빈도를 확인가능
-논문 도용 등을 확인

In [76]:
from ngram import Ngram#file->class
ngram=Ngram()

In [77]:
#2문장(gram)나누기
a="오늘 강남에서 맛있는 스파게티를 먹었다."
b="강남에서 먹었던 오늘의 스파게티는 맛있었다."
print(ngram.ngram(a,2))
print(ngram.ngram(b,2))
#cnn:겹쳐서 찍음:??

['오늘', '늘 ', ' 강', '강남', '남에', '에서', '서 ', ' 맛', '맛있', '있는', '는 ', ' 스', '스파', '파게', '게티', '티를', '를 ', ' 먹', '먹었', '었다', '다.']
['강남', '남에', '에서', '서 ', ' 먹', '먹었', '었던', '던 ', ' 오', '오늘', '늘의', '의 ', ' 스', '스파', '파게', '게티', '티는', '는 ', ' 맛', '맛있', '있었', '었다', '다.']


In [78]:
#3문장으로 나누기(띄어쓰기 포함)
print(ngram.ngram(a,3))
print(ngram.ngram(b,3))

['오늘 ', '늘 강', ' 강남', '강남에', '남에서', '에서 ', '서 맛', ' 맛있', '맛있는', '있는 ', '는 스', ' 스파', '스파게', '파게티', '게티를', '티를 ', '를 먹', ' 먹었', '먹었다', '었다.']
['강남에', '남에서', '에서 ', '서 먹', ' 먹었', '먹었던', '었던 ', '던 오', ' 오늘', '오늘의', '늘의 ', '의 스', ' 스파', '스파게', '파게티', '게티는', '티는 ', '는 맛', ' 맛있', '맛있었', '있었다', '었다.']


In [79]:
#유사도 비교
r2=ngram.diff_ngram(a,b,2)
print("2gram 유사도:",r2)

r3=ngram.diff_ngram(a,b,3)
print("3gram 유사도:",r3)#45%:복사했다고 볼수 있음?

2gram 유사도: (0.7619047619047619, ['오늘', '강남', '남에', '에서', '서 ', ' 맛', '맛있', '는 ', ' 스', '스파', '파게', '게티', ' 먹', '먹었', '었다', '다.'])
3gram 유사도: (0.45, ['강남에', '남에서', '에서 ', ' 맛있', ' 스파', '스파게', '파게티', ' 먹었', '었다.'])


In [80]:
a="머신러닝은 매우 재미있는 기술이라 공부하고 있습니다."#띄어쓰기 중요
b="공부하면 재미있는 기술이라 머신러닝을 배우고 있습니다."
r2=ngram.diff_ngram(a,b,2)
print("2gram 유사도:",r2)

2gram 유사도: (0.75, ['머신', '신러', '러닝', ' 재', '재미', '미있', '있는', '는 ', ' 기', '기술', '술이', '이라', '라 ', '공부', '부하', '고 ', ' 있', '있습', '습니', '니다', '다.'])


In [81]:
a="본문과 전혀 관계 없는 내용이지만 마시멜론ㄴ 맛있습니다."
b="마시멜로는 본문과 전혀 관계 없이 맛있습니다."
r2=ngram.diff_ngram(a,b,2)
print("2gram 유사도:",r2)

2gram 유사도: (0.6333333333333333, ['본문', '문과', '과 ', ' 전', '전혀', '혀 ', ' 관', '관계', '계 ', ' 없', '는 ', '마시', '시멜', ' 맛', '맛있', '있습', '습니', '니다', '다.'])


In [82]:
a="파이썬 프로그래밍에서 중요한 것은 블록 입니다."
b="겨울에는 충분한 수분을 보충해야 합니다."
r2=ngram.diff_ngram(a,b,2)
print("2gram 유사도:",r2)

2gram 유사도: (0.12, ['한 ', '니다', '다.'])


----
#코사인(cosine) 유사도
:두문장을 비교하여 유사도 측정

In [83]:
a="머신러닝은 매우 재미있는 기술이라 공부하고 있습니다."
b="공부하면 재미있는 기술이라 머신러닝을 배우고 있습니다."

from sklearn.feature_extraction.text import TfidfVectorizer
sentences=(a,b)

tfid_vectorizer=TfidfVectorizer()
sentences

('머신러닝은 매우 재미있는 기술이라 공부하고 있습니다.', '공부하면 재미있는 기술이라 머신러닝을 배우고 있습니다.')

In [84]:
#문장 벡터화하기(어휘사전 만들기)=숫자만들기:코사인유사도:많이쓰임
tfid_matrix=tfid_vectorizer.fit_transform(sentences)
tfid_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (2, 9)>

In [85]:
#코사인 유사도
from sklearn.metrics.pairwise import cosine_similarity

#첫번째 문장과 두번째 문장 비교
cos_similar=cosine_similarity(tfid_matrix[0],tfid_matrix[1])
print("코사인 유사도 :", cos_similar)#0.33 높은 편(평균값?)

코사인 유사도 : [[0.33609693]]


In [86]:
# a="머신러닝은 매우 재미있는 기술이라 공부하고 있습니다."
# b="공부하면 재미있는 기술이라 머신러닝을 배우고 있습니다."
tfid_matrix.toarray()#문장 2개:평균값 0.33
#안겹치는 단어 포함해서 점수를 내줌주(0:매우)
#위치를 떠나 비슷한 단어를 1:1 로 비교하며 찾는것 같음:코사인

array([[0.47042643, 0.        , 0.33471228, 0.47042643, 0.47042643,
        0.        , 0.        , 0.33471228, 0.33471228],
       [0.        , 0.47042643, 0.33471228, 0.        , 0.        ,
        0.47042643, 0.47042643, 0.33471228, 0.33471228]])

In [87]:
a="나는 빨간색 딸기 좋아한다."
b="나는 빨간색 체리 좋아한다."
c="나는 빨간색 딸기 빨간색 딸기 좋아한다."
d="파이썬은 재미있는 언어 입니다."

sentences=(a,d)
tfid_vectorizer=TfidfVectorizer()
tfid_matrix=tfid_vectorizer.fit_transform(sentences)
cos_similar=cosine_similarity(tfid_matrix[0],tfid_matrix[1])
cos_similar#ab순서/ac같은단어/ad0프로

array([[0.]])

---
#유사도를 이용한 추천 시스템 구현
:코사인 유사도 만으로 영화의 줄거리에 기반하여 영화를 추천하는 추천 시스템 구축

In [88]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer#딥러닝:벡터화 후 임베딩(순서 중요)
from sklearn.metrics.pairwise import cosine_similarity

In [89]:
data= pd.read_csv("../Data/movies_metadata.csv", low_memory=False)#압축하지 마라
data.head(2)#overview

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [90]:
data.columns#유사도:overview=>title 추천:다음영화

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

코사인 유사도에 사용할 데이터는 영화제목에 해당하는 title,줄거리에 해당하는 overview
좋아하는 영화를 입력하면 해당영화의 줄거리와 유사한 줄거리의 영화를 찾아서 추천하는 시스템을 만들것입니다.

In [91]:
data[['title','overview']].head()

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [92]:
# data[['title','overview']].shape()#45000

In [93]:
#상위 2만개의 샘플
data=data.head(20000)

In [94]:
#overview에 결측치
data['overview'].isnull().sum()

np.int64(135)

In [95]:
#결측치를 빈값으로 대체
data['overview']=data['overview'].fillna('')

In [96]:
tfidf=TfidfVectorizer(stop_words='english')#aaughh?-하품
tfidf_matrix=tfidf.fit_transform(data['overview'])
tfidf_matrix.shape#컬럼 47487

(20000, 47487)

In [97]:
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)#똑같은거 비교 : 반복문
print("코사인 유사도 연산 결과:",cosine_sim.shape)

코사인 유사도 연산 결과: (20000, 20000)


In [98]:
#기존 데이터 프레임으로 부터 영화의 타이틀을 key, 영화의 인덱스를 value로 하는 딕셔너리
from turtle import title


title_to_index=dict(zip(data['title'],data.index))

idx= title_to_index['Father of the Bride Part II']
print(idx)

4


In [101]:
#선택한 영화의 제목을 입력하면 코사인 유사도를 통해 overview가 유사한 10개의 영화를 찾아내는 함수

def get_recommendations(title, cosine_sim=cosine_sim):
    #선택한 영화의 타이틀로 부터 해당 영화의 인덱스를 받아온다.
    idx=title_to_index[title]

    #해당 영화와 모든 영화와의 유사도를 가져온다.
    sim_scores=list(enumerate(cosine_sim[idx]))

    #유사도에 따라 영향들을 정렬한다.
    sim_scores=sorted(sim_scores,key=lambda x:x[1],reverse=True)#나빼고

    #가장 유사한 10개의 영화를 받아온다.
    sim_scores=sim_scores[1:11]

    #가장 유사한 10개의 영화의 인덱스를 얻는다.
    movie_indices=[idx[0] for idx in sim_scores]

    #가장 유사한 10개의 영화의 제목을 리턴한다.
    return data['title'].iloc[movie_indices]


In [102]:
#영화 다크 나이트 라이즈의 overview가 유사한 영화
get_recommendations('The Dark Knight Rises')

12481                            The Dark Knight
150                               Batman Forever
1328                              Batman Returns
15511                 Batman: Under the Red Hood
585                                       Batman
9230          Batman Beyond: Return of the Joker
18035                           Batman: Year One
19792    Batman: The Dark Knight Returns, Part 1
3095                Batman: Mask of the Phantasm
10122                              Batman Begins
Name: title, dtype: str

In [103]:
get_recommendations('Toy Story')

15348               Toy Story 3
2997                Toy Story 2
10301    The 40 Year Old Virgin
8327                  The Champ
1071      Rebel Without a Cause
11399    For Your Consideration
1932                  Condorman
3057            Man on the Moon
485                      Malice
11606              Factory Girl
Name: title, dtype: str

In [105]:
get_recommendations('Jumanji')

6166                       Brainscan
8801                         Quintet
17223                 The Dark Angel
9503                       Word Wars
13601    The Mindscape of Alan Moore
16843                         DeVour
8079                         Masques
6055                Poolhall Junkies
19726                 Wreck-It Ralph
2486                        eXistenZ
Name: title, dtype: str